In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', 50)

In [ ]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())
print("\nColumn data types:\n", df.dtypes)

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
missing_df

In [ ]:
if len(missing_df) > 0:
    plt.figure(figsize=(10, 5))
    plt.bar(missing_df.index.astype(str), missing_df['missing_pct'])
    plt.xticks(rotation=90)
    plt.ylabel("% Missing")
    plt.title("Percentage of Missing Values per Column")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found in the dataset.")

In [ ]:
threshold = 0.5
cols_to_drop = missing_df[missing_df['missing_pct'] > threshold * 100].index.tolist()
df_clean = df.drop(columns=cols_to_drop)
print("Dropped columns (>50% missing):", cols_to_drop)

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
categorical_cols = df_clean.select_dtypes(exclude=[np.number]).columns

for col in numeric_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    if df_clean[col].isnull().any():
        mode_val = df_clean[col].mode()
        if not mode_val.empty:
            df_clean[col] = df_clean[col].fillna(mode_val[0])

print("\nRemaining missing values after cleaning:", df_clean.isnull().sum().sum())
df_clean.head()

In [ ]:
df_clean.describe().T

In [ ]:
if len(categorical_cols) > 0:
    display(df_clean[categorical_cols].describe().T)
else:
    print("No categorical columns present.")

In [ ]:
num_cols = df_clean.select_dtypes(include=[np.number]).columns
n_cols = 3
n_rows = int(np.ceil(len(num_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = np.array(axes).flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df_clean[col].dropna(), bins=30, color='darkorange', edgecolor='black')
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = np.array(axes).flatten()

for i, col in enumerate(num_cols):
    axes[i].boxplot(df_clean[col].dropna(), vert=False)
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
if len(categorical_cols) > 0:
    for col in categorical_cols:
        plt.figure(figsize=(8, 4))
        df_clean[col].value_counts().plot(kind='bar', color='firebrick', edgecolor='black')
        plt.title(f"Distribution of {col}")
        plt.ylabel("Count")
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()

In [ ]:
corr = df_clean[num_cols].corr()

plt.figure(figsize=(10, 8))
im = plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, label='Correlation coefficient')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Matrix (Numeric Features)")
plt.tight_layout()
plt.show()

corr

In [ ]:
corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
top_corr = corr_pairs.abs().sort_values(ascending=False).head(10)
print("Top 10 strongest correlations:")
top_corr